# Save EU PM<sub>2.5</sub> mortality in single file

EU mortality is saved on an annual basis for each mortality outcome. This script combines yearly files into one and calculates the total mortality based on all health outcomes.

In [1]:
import os
import glob
import xarray as xr
import config
from utils.utils import require_dir
import pathlib

In [2]:
# Number of samples
n_samples = 1000

In [3]:
# === Health variables ===
health_vars = ["COPD", "DIABETES", "ISCHEMIC_HEART_DISEASE",
               "LOWER_RESPIRATORY_INFECTIONS", "LUNG_CANCER",
               "STROKE", "DEMENTIA"]

In [10]:
# === Scenario and path config ===
# For RR curves and file name
GBD_version = "GBD23"

scenarios = ["H", "HL", "L", "LN", "M", "ML", "VL"]
years = [2040, 2060, 2080, 2100]

MORT_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "mortality" / "global" / f"{n_samples}_samples")
SAVE_DIR = require_dir(pathlib.Path(config.WORK_ROOT) / "mortality" )

for scenario in scenarios:
    for year in years:
        print(f"Processing {scenario}, year {year}")
        # Find all files for this scenario/year
        in_files = f"EU_mortality_{GBD_version}_*_{n_samples}samples_{scenario}_{year}.nc"
        in_path = os.path.join(MORT_DIR, in_files)
        files = sorted(glob.glob(in_path))

        # Open and combine
        datasets = [xr.open_dataarray(f) for f in files]

        # Align (important in case of slight coordinate mismatches)
        aligned = xr.align(*datasets, join="exact")

        # Sum across the health variables
        summed_da = sum(aligned)

        description = ("Total EU mortality due to PM2.5 "
                       "- scripts by A.F. Wells (2025)")
        summed_da.attrs["description"] = description
        summed_da.attrs["GBD version"] = GBD_version
        summed_da.attrs["scenario"] = scenario
        summed_da.attrs["year"] = year

        out_file = f"EU_mortality_{GBD_version}_{n_samples}samples_{scenario}_{year}.nc"
        out_path = os.path.join(SAVE_DIR, out_file)

        print(f"Saving summed mortality to {out_path}")
        summed_da.to_netcdf(out_path)

print("All processing complete.")

Processing H, year 2040
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23_1000samples_H_2040.nc
Processing H, year 2060
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23_1000samples_H_2060.nc
Processing H, year 2080
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23_1000samples_H_2080.nc
Processing H, year 2100
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23_1000samples_H_2100.nc
Processing HL, year 2040
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23_1000samples_HL_2040.nc
Processing HL, year 2060
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23_1000samples_HL_2060.nc
Processing HL, year 2080
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23_1000samples_HL_2080.nc
Processing HL, year 2100
Saving summed mortality to /glade/work/awells/EU_pm/mortality/EU_mortality_GBD23